In [ ]:
# -------------------------------
# 1. Imports & Path Setup
# -------------------------------
import os
import sys
import pandas as pd
import numpy as np
from autogluon.tabular import TabularPredictor

# Add project root to sys.path
notebook_dir = os.getcwd() 
project_root = os.path.abspath(os.path.join(notebook_dir, "..")) 
if project_root not in sys.path:
    sys.path.append(project_root)

from src.config import Config as cfg
from src.data_loader import load_data, prepare_data

# -------------------------------
# 2. Load & Prepare Data
# -------------------------------
# Using your modular data loader to get the specific training/test sets
X_train, X_test, y_train, y_test = load_data("encoded")
X_train, X_test, y_train_numeric, y_test_numeric, test_ids, num_classes, int_to_label = prepare_data(
    X_train, X_test, y_train, y_test, target=cfg.TARGET, drop_id=True
)






Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.10.18
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Thu, 26 Mar 2026 19:20:28 +0000
CPU Count:          16
Pytorch Version:    2.8.0+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 3.68/3.68 GB
Total GPU Memory:   Free: 3.68 GB, Allocated: 0.00 GB, Total: 3.68 GB
GPU Count:          1
Memory Avail:       1.75 GB / 15.24 GB (11.5%)
Disk Space Avail:   14.30 GB / 215.22 GB (6.6%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1


Number of classes: 2
X_train shape: (300000, 116)
X_test shape: (200000, 116)
y_train shape: (300000,)
y_test labels are not available
Original: (15000, 116)
Sample: (15000, 116)


DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stacking as a consequence.
	This is used to identify the optimal `num_stack_levels` value. Copies of AutoGluon will be fit on subsets of the data. Then holdout validation data is used to detect stacked overfitting.
	Running DyStack for up to 150s of the 600s of remaining time (25%).
DyStack: Disabling memory safe fit mode in DyStack because GPUs were detected and num_gpus='auto' (GPUs cannot be used in memory safe fit mode). If you want to use memory safe fit mode, manually set `num_gpus=0`.
Running DyStack sub-fit ...
Beginning AutoGluon training ... Time limit = 150s
AutoGluon will save models to "/home/ismail/x42/models/AutogluonModels/ds_sub_fit/sub_fit_ho"
Train Data Rows:    13333
Train Data Columns: 116
Label Column:       target
Problem Type:       binary
Preprocessing data ...
Selected class <--> label mapping:  class 1 = 1

AutoGluon Leaderboard:
                          model  score_val eval_metric  pred_time_val  \
0           WeightedEnsemble_L2   0.789805     roc_auc       1.268215   
1           WeightedEnsemble_L3   0.789759     roc_auc       7.625556   
2               CatBoost_BAG_L1   0.787147     roc_auc       0.030934   
3               CatBoost_BAG_L2   0.786697     roc_auc       7.527718   
4          CatBoost_r177_BAG_L2   0.786511     roc_auc       7.535424   
5          CatBoost_r177_BAG_L1   0.785697     roc_auc       0.030179   
6               LightGBM_BAG_L2   0.785213     roc_auc       7.579123   
7             LightGBMXT_BAG_L2   0.784591     roc_auc       7.573686   
8                XGBoost_BAG_L2   0.782683     roc_auc       7.656268   
9           LightGBM_r96_BAG_L1   0.782403     roc_auc       0.668205   
10        NeuralNetTorch_BAG_L2   0.781402     roc_auc       7.873174   
11    NeuralNetTorch_r79_BAG_L2   0.779362     roc_auc       7.761264   
12         LightGBMLarge_BAG

In [ ]:

# AutoGluon requires a single DataFrame for training
train_data = X_train.copy()
train_data[cfg.TARGET] = y_train_numeric

# -------------------------------
# 3. Define Params from Config
# -------------------------------
# Mapping your src.config params to AutoGluon's fit arguments
ag_params = {
    'label': cfg.TARGET,
    'problem_type': 'binary' if num_classes <= 2 else 'multiclass',
    'eval_metric': cfg.KAGGLE_EVAL
}
# -------------------------------
# 4. Train AutoGluon Predictor
# -------------------------------
predictor = TabularPredictor(
    label=ag_params['label'],
    problem_type=ag_params['problem_type'],
    eval_metric=ag_params['eval_metric'],
    path="AutogluonModels/" 
).fit(
    train_data=train_data,
    time_limit=600,               
    presets='best_quality',        
    # random_state=cfg.RANDOM_STATE,  <-- REMOVE OR COMMENT THIS LINE
    verbosity=2
)

# -------------------------------
# 5. Predict & Submit
# -------------------------------
# AutoGluon handles the internal feature engineering for X_test automatically
if cfg.SUBMIT_PROBABILITIES:
    # Get probabilities for all classes
    predictions = predictor.predict_proba(X_test)
    
    # If binary, usually just want the positive class probability
    if ag_params['problem_type'] == 'binary':
        predictions = predictions.iloc[:, 1]
else:
    # Get hard labels
    predictions = predictor.predict(X_test)

# -------------------------------
# 6. Save using Config Paths
# -------------------------------
submission = pd.DataFrame({
    cfg.ID: test_ids,
    cfg.TARGET: predictions
})

submission.to_csv('../outputs/submissions/autoglean.csv', index=False)
print(f"AutoGluon Leaderboard:\n{predictor.leaderboard(silent=True)}")
print(f"Saved submission to: {'../outputs/submissions/autoglean.csv'}")

<mark>2️⃣ Text features (raw text like reviews, comments)

If you have long free-form text (e.g., product reviews, tweets), AutoGluon can handle it too, but you need to tell it explicitly you have a text column.

It will automatically create text models (like bag-of-words, embeddings) and ensemble them with other models.

</mark>

If you want, I can make a ready-to-run AutoGluon template for your binary string target + categorical features dataset?

In [6]:

# -----------------------------
# 6️⃣ Check training summary
# -----------------------------
predictor.fit_summary()



*** Summary of fit() ***
Estimated performance of each model:
                          model  score_val eval_metric  pred_time_val    fit_time  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0           WeightedEnsemble_L2   0.789805     roc_auc       1.268215   90.180412                0.001515           0.571481            2       True         18
1           WeightedEnsemble_L3   0.789759     roc_auc       7.625556  284.171649                0.003284           1.881366            3       True         32
2               CatBoost_BAG_L1   0.787147     roc_auc       0.030934   14.947376                0.030934          14.947376            1       True          5
3               CatBoost_BAG_L2   0.786697     roc_auc       7.527718  278.324578                0.035442          12.187269            2       True         23
4          CatBoost_r177_BAG_L2   0.786511     roc_auc       7.535424  277.284657                0.043149          11.147348            2 

/home/ismail/miniconda/envs/ml-dl/lib/python3.10/site-packages/autogluon/core/utils/plots.py:169: UserWarning: AutoGluon summary plots cannot be created because bokeh is not installed. To see plots, please do: "pip install bokeh==2.0.1"
  warnings.warn('AutoGluon summary plots cannot be created because bokeh is not installed. To see plots, please do: "pip install bokeh==2.0.1"')


{'model_types': {'LightGBMXT_BAG_L1': 'StackerEnsembleModel_LGB',
  'LightGBM_BAG_L1': 'StackerEnsembleModel_LGB',
  'RandomForestGini_BAG_L1': 'StackerEnsembleModel_RF',
  'RandomForestEntr_BAG_L1': 'StackerEnsembleModel_RF',
  'CatBoost_BAG_L1': 'StackerEnsembleModel_CatBoost',
  'ExtraTreesGini_BAG_L1': 'StackerEnsembleModel_XT',
  'ExtraTreesEntr_BAG_L1': 'StackerEnsembleModel_XT',
  'NeuralNetFastAI_BAG_L1': 'StackerEnsembleModel_NNFastAiTabular',
  'XGBoost_BAG_L1': 'StackerEnsembleModel_XGBoost',
  'NeuralNetTorch_BAG_L1': 'StackerEnsembleModel_TabularNeuralNetTorch',
  'LightGBMLarge_BAG_L1': 'StackerEnsembleModel_LGB',
  'CatBoost_r177_BAG_L1': 'StackerEnsembleModel_CatBoost',
  'NeuralNetTorch_r79_BAG_L1': 'StackerEnsembleModel_TabularNeuralNetTorch',
  'LightGBM_r131_BAG_L1': 'StackerEnsembleModel_LGB',
  'NeuralNetFastAI_r191_BAG_L1': 'StackerEnsembleModel_NNFastAiTabular',
  'CatBoost_r9_BAG_L1': 'StackerEnsembleModel_CatBoost',
  'LightGBM_r96_BAG_L1': 'StackerEnsembleMod

In [7]:

cv_results = predictor.leaderboard(silent=True)
print(cv_results)

                          model  score_val eval_metric  pred_time_val  \
0           WeightedEnsemble_L2   0.789805     roc_auc       1.268215   
1           WeightedEnsemble_L3   0.789759     roc_auc       7.625556   
2               CatBoost_BAG_L1   0.787147     roc_auc       0.030934   
3               CatBoost_BAG_L2   0.786697     roc_auc       7.527718   
4          CatBoost_r177_BAG_L2   0.786511     roc_auc       7.535424   
5          CatBoost_r177_BAG_L1   0.785697     roc_auc       0.030179   
6               LightGBM_BAG_L2   0.785213     roc_auc       7.579123   
7             LightGBMXT_BAG_L2   0.784591     roc_auc       7.573686   
8                XGBoost_BAG_L2   0.782683     roc_auc       7.656268   
9           LightGBM_r96_BAG_L1   0.782403     roc_auc       0.668205   
10        NeuralNetTorch_BAG_L2   0.781402     roc_auc       7.873174   
11    NeuralNetTorch_r79_BAG_L2   0.779362     roc_auc       7.761264   
12         LightGBMLarge_BAG_L2   0.779195     roc_

In [6]:
pd.set_option('display.float_format', lambda x: f'{x:.5f}')
print(cv_results)

                      model  score_val eval_metric  pred_time_val  fit_time  \
0       WeightedEnsemble_L3    0.89466     roc_auc       30.46668 306.91689   
1       WeightedEnsemble_L2    0.89400     roc_auc       10.99636 212.88074   
2           LightGBM_BAG_L2    0.89386     roc_auc       21.32197 255.89448   
3         LightGBMXT_BAG_L2    0.89349     roc_auc       22.10833 259.58725   
4           LightGBM_BAG_L1    0.89341     roc_auc        3.14432  19.44231   
5           CatBoost_BAG_L2    0.89339     roc_auc       20.89375 272.56270   
6           CatBoost_BAG_L1    0.89317     roc_auc        0.06628 159.59970   
7         LightGBMXT_BAG_L1    0.89147     roc_auc        4.38115  24.76242   
8   RandomForestEntr_BAG_L2    0.89101     roc_auc       25.27261 272.04202   
9   RandomForestGini_BAG_L2    0.89030     roc_auc       25.49758 272.21767   
10    ExtraTreesEntr_BAG_L1    0.88434     roc_auc        3.50567   9.19667   
11  RandomForestGini_BAG_L1    0.88380     roc_auc  

In [ ]:
prediction = predictor.predict_proba(X_test)

data_submit = pd.read_csv('../sample_submission.csv')
data_submit.Exited = prediction[1]
data_submit[['id', 'Exited']].to_csv('../outputs/submissions/autoglean.csv', index=False)


Perfect — your code already works, but we can make it **“advanced”** in several ways to **maximize performance, get more insights, and make it production-ready**. Here’s how:

---

## **1️⃣ Explicit Hyperparameter Tuning**

Instead of using default models, you can define **which models to train** and their **hyperparameters**, including boosting parameters, neural nets, and stacking options:

```python
hyperparameters = {
    'GBM': {'num_boost_round': 500, 'learning_rate': 0.05, 'num_leaves': 31},
    'CAT': {'iterations': 500, 'learning_rate': 0.05, 'depth': 6},
    'XGB': {'n_estimators': 500, 'learning_rate': 0.05, 'max_depth': 6},
    'RF': {'n_estimators': 200, 'max_depth': None},
    'NN_TORCH': {'epochs': 20, 'learning_rate': 0.001},
}
```

Then pass it to `fit()`:

```python
predictor = TabularPredictor(label='target', eval_metric='accuracy').fit(
    train_data=train_data,
    time_limit=1200,  # increase time if dataset is small
    hyperparameters=hyperparameters,
    presets='best_quality',
    num_bag_folds=5,  # enables k-fold bagging for robust CV
    stack_ensemble_levels=2  # create a two-level stacking ensemble
)
```

---

## **2️⃣ Enable K-Fold Bagging & Stacking**

* **Bagging (`num_bag_folds`)** improves stability on small datasets.
* **Stacked ensembles (`stack_ensemble_levels`)** improve overall accuracy by combining multiple models.

---

## **3️⃣ Handle Categorical/Text Explicitly**

If you have string/categorical features or text, AutoGluon usually detects them automatically, but you can be explicit:

```python
feature_metadata = {
    'categorical': cat_cols,
    'text': text_cols if 'text_cols' in locals() else []
}

predictor = TabularPredictor(label='target').fit(
    train_data=train_data,
    feature_metadata=feature_metadata
)
```

---

## **4️⃣ Save & Reload Models**

* Save the predictor to reuse later without retraining:

```python
predictor.save('autogluon_model')
# reload later
# predictor = TabularPredictor.load('autogluon_model')
```

---

## **5️⃣ Get Out-of-Fold (OOF) Predictions**

This is useful if you want to **stack your own models or analyze CV performance**:

```python
oof_preds = predictor.get_oof_pred_proba()
```

---

## **6️⃣ Evaluate & Compare Models**

* `leaderboard()` shows **all trained models, validation scores, hyperparameters**:

```python
cv_results = predictor.leaderboard(train_data, extra_metrics=['f1', 'roc_auc'])
print(cv_results)
```

* You can sort by any metric to see the **best performing model**.

---

## **7️⃣ Make Probabilistic Predictions**

Instead of just class labels, get probabilities for each class:

```python
proba_preds = predictor.predict_proba(X_test)
```

---

## **8️⃣ Advanced Prediction Example**

```python
predictions = predictor.predict(X_test)
proba_preds = predictor.predict_proba(X_test)

# save both
predictions.to_csv('submission.csv', index=False)
proba_preds.to_csv('submission_proba.csv', index=False)
```

---

✅ **Summary of Advanced Features**

| Feature                               | Why Use It                             |
| ------------------------------------- | -------------------------------------- |
| Hyperparameters dict                  | Customize models for better accuracy   |
| num_bag_folds & stack_ensemble_levels | More robust, better generalization     |
| feature_metadata                      | Explicit categorical/text handling     |
| OOF predictions                       | Useful for stacking or analysis        |
| predict_proba                         | Useful for probability-based scoring   |
| save/load model                       | Reuse trained models, avoid retraining |
| leaderboard with extra_metrics        | Compare models using multiple metrics  |

---

If you want, I can **rewrite your notebook fully “advanced”** with all these features applied **ready to run on your X_train/X_test/y_train dataset**.

Do you want me to do that?
